In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("filtered_portion_ready_foods.csv")



In [2]:
def clean_numeric_column(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
        .replace(["-", "", "nan", "None"], np.nan)
        .pipe(pd.to_numeric, errors="coerce")
    )

In [3]:
import pandas as pd
import numpy as np

# Assume your merged dataframe is named df
food_candidates = df.copy()

# ============================================================
# 1. Make sure gram_per_portion is numeric
# ============================================================

food_candidates["gram_per_portion"] = pd.to_numeric(
    food_candidates["gram_per_portion"],
    errors="coerce"
)

# Remove rows without valid portion size
food_candidates = food_candidates.dropna(subset=["gram_per_portion"]).copy()
food_candidates = food_candidates[food_candidates["gram_per_portion"] > 0].copy()


# ============================================================
# 2. Convert nutrient columns to numeric
# ============================================================

numeric_cols = [
    "gram_per_portion",
    "energy_kcal_100g",
    "protein_g_100g",
    "fat_g_100g",
    "carb_g_100g",
    "sodium_mg_100g",
    "potassium_mg_100g",
    "calcium_mg_100g",
    "fiber_g_100g"
]


for col in numeric_cols:
    if col in food_candidates.columns:
        food_candidates[col] = clean_numeric_column(food_candidates[col])


# ============================================================
# 3. Create nutrient-per-portion features
# ============================================================

food_candidates["energy_kcal_per_portion"] = (
    food_candidates["energy_kcal_100g"] *
    food_candidates["gram_per_portion"] / 100
)

food_candidates["protein_g_per_portion"] = (
    food_candidates["protein_g_100g"] *
    food_candidates["gram_per_portion"] / 100
)

food_candidates["fat_g_per_portion"] = (
    food_candidates["fat_g_100g"] *
    food_candidates["gram_per_portion"] / 100
)

food_candidates["carb_g_per_portion"] = (
    food_candidates["carb_g_100g"] *
    food_candidates["gram_per_portion"] / 100
)

food_candidates["sodium_mg_per_portion"] = (
    food_candidates["sodium_mg_100g"] *
    food_candidates["gram_per_portion"] / 100
)

food_candidates["potassium_mg_per_portion"] = (
    food_candidates["potassium_mg_100g"] *
    food_candidates["gram_per_portion"] / 100
)

food_candidates["calcium_mg_per_portion"] = (
    food_candidates["calcium_mg_100g"] *
    food_candidates["gram_per_portion"] / 100
)

food_candidates["fiber_g_per_portion"] = (
    food_candidates["fiber_g_100g"] *
    food_candidates["gram_per_portion"] / 100
)

In [4]:
check_cols = [
    "food_code",
    "food_name",
    "protein_g_100g",
    "gram_per_portion",
    "protein_g_per_portion"
]

display(
    food_candidates.loc[
        food_candidates["protein_g_per_portion"] == 0,
        check_cols
    ].head(100)
)

,food_code,food_name,protein_g_100g,gram_per_portion,protein_g_per_portion
100,KR009,Minyak kacang tanah,0.0,5.0,0.0
101,KR010,Minyak kedelai,0.0,5.0,0.0
103,KR014,Minyak zaitun,0.0,5.0,0.0
107,MP014,Sirup,0.0,15.0,0.0


In [5]:
df_check = df.copy()

df_check["protein_original"] = df_check["protein_g_100g"]
df_check["protein_after_clean"] = clean_numeric_column(df_check["protein_g_100g"])

display(
    df_check.loc[
        df_check["protein_original"].astype(str).str.contains(",", na=False),
        ["food_code", "food_name", "protein_original", "protein_after_clean"]
    ].head(30)
)

,food_code,food_name,protein_original,protein_after_clean
11,BR013,"Kentang, segar","2,1",2.1
12,BR014,"Kentang hitam, segar","0,9",0.9
13,CR018,"Kacang kedelai, segar","30,2",30.2
15,CP005,"kacang hijau, rebus","8,7",8.7
16,CP007,"Kacang kedelai, rebus","20,2",20.2
17,CP037,Kembang tahu,"48,9",48.9
19,CP061,"Tahu, mentah","10,9",10.9
20,CP077,"Tempe kedelai, mentah","20,8",20.8
21,DR008,"Bayam, segar","0,9",0.9
22,DR009,"Bayam merah, segar","2,2",2.2


In [6]:
required_for_milp = [
    "food_code",
    "food_name",
    "category_code",
    "category_name",
    "urt",
    "gram_per_portion",
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

missing_cols = [
    col for col in required_for_milp 
    if col not in food_candidates.columns
]

print("Missing columns:", missing_cols)

display(food_candidates[required_for_milp].head(5))

Missing columns: []


,food_code,food_name,category_code,category_name,urt,gram_per_portion,energy_kcal_per_portion,protein_g_per_portion,fat_g_per_portion,carb_g_per_portion,sodium_mg_per_portion,potassium_mg_per_portion,calcium_mg_per_portion,fiber_g_per_portion
0,AR001,"Beras giling, mentah",MP,Makanan Pokok,3/4 Gelas,100.0,357.0,8.4,1.7,77.1,27.0,71.0,147.0,0.2
1,AR002,"Beras giling var pelita, mentah",MP,Makanan Pokok,3/4 Gelas,100.0,369.0,9.5,1.4,77.1,0.0,34.0,68.0,0.4
2,AR003,"Beras giling var rojolele, mentah",MP,Makanan Pokok,3/4 Gelas,100.0,357.0,8.4,1.7,77.1,112.9,34.0,147.0,0.2
3,AR004,"Beras hitam, mentah",MP,Makanan Pokok,3/4 Gelas,100.0,351.0,8.0,1.3,76.9,15.0,105.0,6.0,20.1
4,AR013,"Beras tumbuk merah, mentah",MP,Makanan Pokok,3/4 Gelas,100.0,352.0,7.3,0.9,76.2,10.0,202.0,15.0,0.8


In [7]:
output_file = "final_data_frame.csv"

food_candidates.to_csv(output_file, index=False, encoding="utf-8-sig")